# AAT6153 Chapter 2 — Hands-on 4
## Sentiment Analysis with LSTM (IMDB)
**Slides:** Processing Sequence Data (68–69), Hands-on 4 (70–74)

Same IMDB task as Hands-on 2, but now the model reads each review **word by word, in order** (many-to-one).
You will train an **LSTM** and a **stacked bidirectional LSTM**, then compare them with the Hands-on 2 models.

*Runtime: Runtime ▸ Change runtime type ▸ **T4 GPU** required.*

---
## Step 1: Load and pad the IMDB reviews (slide 70)

In [ ]:
import numpy as np
import keras
from keras import layers
from keras.datasets import imdb
from keras.utils import pad_sequences

keras.utils.set_random_seed(7)
vocab_size, maxlen = 10000, 200
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=vocab_size)
# post-padding keeps the GPU (cuDNN) LSTM kernel available with masking
x_train = pad_sequences(x_train, maxlen=maxlen, padding="post", truncating="post")
x_test  = pad_sequences(x_test,  maxlen=maxlen, padding="post", truncating="post")
print(x_train.shape, x_test.shape)

## Step 2: Define the LSTM model (slide 71)

In [ ]:
model = keras.Sequential([
    keras.Input(shape=(maxlen,)),
    layers.Embedding(vocab_size, 64, mask_zero=True),  # ignore padding
    layers.LSTM(64),                                   # many-to-one
    layers.Dropout(0.5),
    layers.Dense(1, activation="sigmoid")])
model.compile(optimizer="adam", loss="binary_crossentropy",
              metrics=["accuracy"])
model.summary()

**Check:** Embedding 10,000 × 64 = 640,000; LSTM 4 × 64 × (64 + 64 + 1) = 33,024; Dense 65 → **673,089** parameters.

## Step 3: Train and evaluate (slide 72)

In [ ]:
es = keras.callbacks.EarlyStopping(monitor="val_loss", patience=2,
                                   restore_best_weights=True)
history = model.fit(x_train, y_train, epochs=10, batch_size=128,
                    validation_split=0.2, callbacks=[es])
loss, acc = model.evaluate(x_test, y_test)
print(f"Test accuracy: {acc:.2%}")

In [ ]:
import matplotlib.pyplot as plt

def plot_history(history, title):
    h = history.history
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(h["accuracy"], "b", label="train"); ax1.plot(h["val_accuracy"], "g", label="val")
    ax1.set(xlabel="Epoch", ylabel="Accuracy", title=f"{title}: accuracy"); ax1.legend()
    ax2.plot(h["loss"], "b", label="train"); ax2.plot(h["val_loss"], "g", label="val")
    ax2.set(xlabel="Epoch", ylabel="Loss", title=f"{title}: loss"); ax2.legend()
    plt.show()

plot_history(history, "Embedding + LSTM")

## Step 4: Try a stacked bidirectional LSTM (slide 73)
The first bidirectional layer returns the full sequence so the second can read it. Each `Bidirectional(LSTM(64))` outputs 128 values per step (64 forward + 64 backward).

In [ ]:
bi_model = keras.Sequential([
    keras.Input(shape=(maxlen,)),
    layers.Embedding(vocab_size, 64, mask_zero=True),
    layers.Bidirectional(layers.LSTM(64, return_sequences=True)),
    layers.Bidirectional(layers.LSTM(32)),
    layers.Dropout(0.5),
    layers.Dense(1, activation="sigmoid")])
bi_model.compile(optimizer="adam", loss="binary_crossentropy",
                 metrics=["accuracy"])
bi_history = bi_model.fit(x_train, y_train, epochs=10, batch_size=128,
                          validation_split=0.2, callbacks=[es])
print(bi_model.evaluate(x_test, y_test))

In [ ]:
# Fill in the slide 72 table (copy the two Hands-on 2 numbers from that notebook)
bi_acc = bi_model.evaluate(x_test, y_test, verbose=0)[1]
print(f"{'Model':42s} Test accuracy")
print(f"{'Bag-of-words + Dense (Hands-on 2)':42s} (from Hands-on 2)")
print(f"{'Embedding + average pooling (Hands-on 2)':42s} (from Hands-on 2)")
print(f"{'Embedding + LSTM':42s} {acc:.2%}")
print(f"{'Embedding + stacked bidirectional LSTM':42s} {bi_acc:.2%}")

On IMDB a well-tuned bag-of-words model is a strong baseline, so the LSTM does not always win. A few key words often decide the sentiment.

---
## Step 5: Predict the sentiment of your own reviews (slide 74)
New text must be encoded exactly like the training data: same word index, the **+3 offset**, start token **1**, OOV token **2**, then post-padding to 200.

In [ ]:
word_index = imdb.get_word_index()

def encode(text):
    ids = [1]                                      # 1 = <start>
    for w in text.lower().split():
        i = word_index.get(w, vocab_size) + 3
        ids.append(i if i < vocab_size else 2)     # 2 = <oov>
    return pad_sequences([ids], maxlen=maxlen, padding="post")

for review in ["a wonderful film with great acting",
               "a boring movie and an awful script"]:
    p = model.predict(encode(review), verbose=0)[0, 0]
    print(f"{p:.3f}  {review}")

### Exercise: negation (slide 74)
Write reviews with negation, e.g. *"not good at all"* or *"i did not hate it"*. Does the LSTM handle them better than a bag-of-words model would?

In [ ]:
my_reviews = ["not good at all",
              "i did not hate this movie",
              "the acting was not bad but the story was terrible"]
for review in my_reviews:
    p_lstm = model.predict(encode(review), verbose=0)[0, 0]
    p_bi = bi_model.predict(encode(review), verbose=0)[0, 0]
    print(f"LSTM {p_lstm:.3f}   BiLSTM {p_bi:.3f}   {review}")

---
## Try it yourself
1. Replace `LSTM(64)` with `GRU(64)` (slide 62). Compare accuracy, parameter count and training time per epoch.
2. Replace it with `SimpleRNN(64)`. Why does it struggle with 200-word reviews? (slide 56)
3. Change `padding="post"` to `"pre"` and time one epoch on the GPU. Why is it slower? (slide 10 notes)
4. Add `recurrent_dropout=0.2` to the LSTM (slide 66) and time one epoch.

In [ ]:
# Use this cell for your experiments